In [ ]:
!pip install -q transformers gradio captum

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 18.6 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import torch.nn.functional as F

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)

import gradio as gr

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import re

In [ ]:
model_path = "/content/drive/MyDrive/NLP/distilbert_clickbait_model"

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = AutoModelForSequenceClassification.from_pretrained(model_path)

model.eval()

print("Model loaded successfully!")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
label_names = ["Non-clickbait", "Clickbait"]

def predict(headline):

    if headline.strip() == "":
        return (
            "⚠️ Please enter a headline!",
            "",
            0,
            0
        )

    inputs = tokenizer(
        headline,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=64
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probs = F.softmax(outputs.logits, dim=1)[0]

    pred = torch.argmax(probs).item()

    prediction = label_names[pred]

    confidence = probs[pred].item() * 100

    clickbait_prob = probs[1].item() * 100
    non_clickbait_prob = probs[0].item() * 100

    if prediction == "Clickbait":
        result = "🔴 Clickbait"
    else:
        result = "🟢 Non-clickbait"

    return (
        result,
        f"{confidence:.2f}%",
        clickbait_prob,
        non_clickbait_prob
    )

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown(
        """
        # 📰 Clickbait Headline Detection

        ### DistilBERT-based English Clickbait Classifier
        """
    )

    headline = gr.Textbox(
        label="Enter Headline",
        lines=3,
        placeholder="Example: You Won't Believe What Happened Next..."
    )

    predict_btn = gr.Button(
        "🔍 Predict",
        variant="primary"
    )

    prediction = gr.Textbox(label="Prediction")

    confidence = gr.Textbox(label="Confidence")

    clickbait = gr.Slider(
        minimum=0,
        maximum=100,
        label="Clickbait Probability",
        interactive=False
    )

    non_clickbait = gr.Slider(
        minimum=0,
        maximum=100,
        label="Non-clickbait Probability",
        interactive=False
    )

    predict_btn.click(
        predict,
        inputs=headline,
        outputs=[
            prediction,
            confidence,
            clickbait,
            non_clickbait
        ]
    )

/tmp/ipykernel_411/1788804830.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


In [ ]:
//clickbait
The Unbelievable Stats Behind How Cristiano Ronaldo Scored 50 Goals at Age 41
You Won’t Believe How This New OpenAI Model Passed the Bar Exam with a Perfect Score
This Secret Feature On Your iPhone Is Draining Your Battery Lifespan
This Popular Social Media App Is Quietly Accessing Your Camera In The Background
Five everyday foods that are slowly damaging your brain health
New Study Reveals The Secret Behind Better Sleep
Scientists Discover Something In The Amazon That Will Shock You
//non-clickbait
Inside the Secret AI Algorithm That Boosted Amazon’s Sales by 40% Overnight
How a Single Line of Code Cost a Major Crypto Exchange $500 Million in Less Than 5 Minutes
"Is the Era of Traditional Software Engineers Officially Over? What the Data Suggests"
Scientists discover new species in Amazon rainforest



In [ ]:
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://43d14266516415fc91.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
